In [1]:

from google.colab import drive

drive.mount('/content/drive')


Mounted at /content/drive


In [2]:

import csv

file_path = "/content/drive/MyDrive/CareLoop_NoShow_Guard/data/appointments.csv"

with open(file_path, "r", newline="", encoding="utf-8-sig") as file:
    reader = csv.DictReader(file)
    rows = list(reader)

    print("Column names:", reader.fieldnames)
    print("Total rows:", len(rows))


Column names: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Total rows: 1020


In [3]:

def parse_row(row):
    row["age"] = int(row["age"])
    row["distance_km"] = float(row["distance_km"])
    row["fee_inr"] = int(row["fee_inr"])
    row["prior_no_shows"] = int(row["prior_no_shows"])

    return row


In [4]:

accepted_rows = []
rejected_rows = []

for row in rows:
    try:
        parsed_row = parse_row(row.copy())
        accepted_rows.append(parsed_row)

    except ValueError:
        rejected_rows.append(row)

print("Accepted rows:", len(accepted_rows))
print("Rejected rows:", len(rejected_rows))


Accepted rows: 988
Rejected rows: 32


In [5]:

import json

output_path = "/content/drive/MyDrive/CareLoop_NoShow_Guard/rejected_rows.json"

with open(output_path, "w", encoding="utf-8") as file:
    json.dump(rejected_rows, file, indent=4)

print("Rejected rows saved successfully!")


Rejected rows saved successfully!


In [6]:

import pandas as pd

file_path = "/content/drive/MyDrive/CareLoop_NoShow_Guard/data/appointments.csv"

df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)

df.head()


Dataset shape: (1020, 14)


,appointment_id,patient_id,clinic_id,booked_on,appt_date,age,gender,distance_km,fee_inr,booking_channel,visit_type,reminder_sent,prior_no_shows,status
0,A0001,P509,C05,2025-01-24,2025-01-25,60.0,F,18.0,850,Phone,Consultation,Yes,1,No-Show
1,A0002,P046,C07,2025-05-27,2025-06-06,39.0,M,12.5,350,Walk-in,Follow-up,No,2,Attended
2,A0003,P009,C02,2025-03-24,2025-03-30,63.0,F,4.1,850,Walk-in,Consultation,No,0,Attended
3,A0003,P009,C02,2025-03-24,2025-03-30,63.0,F,4.1,850,Walk-in,Consultation,No,0,Attended
4,A0004,P482,C01,2025-04-18,2025-04-22,52.0,F,7.5,750,App,Consultation,Yes,0,Attended


In [7]:

# Display column information and data types
df.info()

# Count missing values in each column
print("\nMissing values:")
print(df.isnull().sum())

# Count duplicate rows
print("\nDuplicate rows:", df.duplicated().sum())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+ KB

Missing values:
appointment_id      0
patient_id          0
clinic_id           0
booked_on

In [8]:

# Remove completely identical duplicate rows
df = df.drop_duplicates().copy()

print("Rows after removing duplicates:", len(df))


Rows after removing duplicates: 1000


In [9]:

print(df["booking_channel"].value_counts(dropna=False))


booking_channel
App         358
Phone       342
Walk-in     230
APP          10
Walk-in      10
app           9
PHONE         7
 walk-in      5
App           5
Walk-In       5
phone         5
 App          5
 phone        4
WALK-IN       3
Phone         1
walk-in       1
Name: count, dtype: int64


In [10]:

df["booking_channel"] = (
    df["booking_channel"]
    .str.strip()
    .str.title()
)

print(df["booking_channel"].value_counts())


booking_channel
App        387
Phone      359
Walk-In    254
Name: count, dtype: int64


In [11]:

df["booked_on"] = pd.to_datetime(df["booked_on"], errors="coerce")
df["appt_date"] = pd.to_datetime(df["appt_date"], errors="coerce")

print(df[["booked_on", "appt_date"]].dtypes)

print("\nMissing dates after conversion:")
print(df[["booked_on", "appt_date"]].isnull().sum())


booked_on    datetime64[ns]
appt_date    datetime64[ns]
dtype: object

Missing dates after conversion:
booked_on    0
appt_date    6
dtype: int64


In [12]:

invalid_rows = df[
    df["appt_date"].isna() | df["status"].isna()
]

print("Rows with missing appointment date or status:", len(invalid_rows))

display(invalid_rows[["appointment_id", "appt_date", "status"]])


Rows with missing appointment date or status: 10


,appointment_id,appt_date,status
98,A0094,NaT,No-Show
167,A0162,NaT,Attended
199,A0194,2025-04-10,NaN
303,A0296,NaT,Attended
445,A0436,2025-03-08,NaN
504,A0494,NaT,Attended
534,A0523,NaT,Attended
562,A0551,2025-01-02,NaN
678,A0666,NaT,Attended
739,A0725,2025-06-02,NaN


In [13]:

df = df.dropna(subset=["appt_date", "status"]).copy()

print("Rows after cleaning:", len(df))
print("Remaining missing dates:", df["appt_date"].isna().sum())
print("Remaining missing statuses:", df["status"].isna().sum())


Rows after cleaning: 990
Remaining missing dates: 0
Remaining missing statuses: 0


In [14]:

import numpy as np

# Find ages outside the valid range
invalid_age = (df["age"] < 1) | (df["age"] > 100)

print("Unrealistic ages:", invalid_age.sum())

# Replace unrealistic ages with NaN
df.loc[invalid_age, "age"] = np.nan

print("Total missing ages:", df["age"].isna().sum())


Unrealistic ages: 5
Total missing ages: 17


In [15]:

# Calculate the median age
median_age = df["age"].median()

print("Median age:", median_age)

# Replace missing ages with the rounded median
df["age"] = df["age"].fillna(round(median_age)).astype(int)

print("Missing ages after filling:", df["age"].isna().sum())


Median age: 48.0
Missing ages after filling: 0


In [16]:

# Check missing distances
print("Missing distances before:", df["distance_km"].isna().sum())

# Calculate median distance
median_distance = df["distance_km"].median()
print("Median distance:", median_distance)

# Fill missing distances
df["distance_km"] = df["distance_km"].fillna(median_distance)

print("Missing distances after:", df["distance_km"].isna().sum())


Missing distances before: 20
Median distance: 8.25
Missing distances after: 0


In [17]:

Q1 = df["distance_km"].quantile(0.25)
Q3 = df["distance_km"].quantile(0.75)

IQR = Q3 - Q1

lower_fence = Q1 - 1.5 * IQR
upper_fence = Q3 + 1.5 * IQR

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower fence:", lower_fence)
print("Upper fence:", upper_fence)

outliers = (
    (df["distance_km"] < lower_fence) |
    (df["distance_km"] > upper_fence)
)

print("Distance outliers:", outliers.sum())


Q1: 4.625
Q3: 13.0
IQR: 8.375
Lower fence: -7.9375
Upper fence: 25.5625
Distance outliers: 26


In [18]:

# Cap distance values using IQR fences
df["distance_km"] = np.clip(
    df["distance_km"],
    lower_fence,
    upper_fence
)

# Round distances to 1 decimal place
df["distance_km"] = df["distance_km"].round(1)

print("Maximum distance after capping:", df["distance_km"].max())
print("Minimum distance:", df["distance_km"].min())
print("Missing distances:", df["distance_km"].isna().sum())


Maximum distance after capping: 25.6
Minimum distance: 0.6
Missing distances: 0


In [19]:

# Calculate quartiles for appointment fees
fee_Q1 = df["fee_inr"].quantile(0.25)
fee_Q3 = df["fee_inr"].quantile(0.75)

fee_IQR = fee_Q3 - fee_Q1

fee_lower = fee_Q1 - 1.5 * fee_IQR
fee_upper = fee_Q3 + 1.5 * fee_IQR

print("Q1:", fee_Q1)
print("Q3:", fee_Q3)
print("IQR:", fee_IQR)
print("Lower fence:", fee_lower)
print("Upper fence:", fee_upper)

fee_outliers = (
    (df["fee_inr"] < fee_lower) |
    (df["fee_inr"] > fee_upper)
)

print("Fee outliers:", fee_outliers.sum())


Q1: 500.0
Q3: 787.5
IQR: 287.5
Lower fence: 68.75
Upper fence: 1218.75
Fee outliers: 3


In [20]:

# Cap fees using the IQR fences
df["fee_inr"] = np.clip(
    df["fee_inr"],
    fee_lower,
    fee_upper
)

# Round fees and convert to integers
df["fee_inr"] = df["fee_inr"].round().astype(int)

print("Maximum fee:", df["fee_inr"].max())
print("Minimum fee:", df["fee_inr"].min())
print("Missing fees:", df["fee_inr"].isna().sum())


Maximum fee: 1219
Minimum fee: 300
Missing fees: 0


In [21]:

# Calculate days between booking and appointment
df["lead_days"] = (
    df["appt_date"] - df["booked_on"]
).dt.days

print("Minimum lead days:", df["lead_days"].min())
print("Maximum lead days:", df["lead_days"].max())

df[["booked_on", "appt_date", "lead_days"]].head()


Minimum lead days: 1
Maximum lead days: 45


,booked_on,appt_date,lead_days
0,2025-01-24,2025-01-25,1
1,2025-05-27,2025-06-06,10
2,2025-03-24,2025-03-30,6
4,2025-04-18,2025-04-22,4
5,2024-11-23,2025-01-02,40


In [22]:

# Convert appointment status into a binary target
df["no_show"] = (df["status"] == "No-Show").astype(int)

print(df["no_show"].value_counts())

# Calculate overall no-show percentage
no_show_rate = df["no_show"].mean() * 100

print("No-show rate:", round(no_show_rate, 1), "%")


no_show
0    772
1    218
Name: count, dtype: int64
No-show rate: 22.0 %


In [23]:

# Select four numerical columns
X = df[["age", "distance_km", "fee_inr", "lead_days"]].to_numpy()

print("Array shape:", X.shape)

# Calculate mean of each column
means = np.mean(X, axis=0)

print("Mean values:", np.round(means, 2))


Array shape: (990, 4)
Mean values: [ 48.17   9.5  635.71  11.82]


In [24]:

# Calculate population standard deviation
std_devs = np.std(X, axis=0)

print("Standard deviations:", np.round(std_devs, 2))

# Calculate Z-scores
z_scores = (X - means) / std_devs

# Select distance column (index 1)
distance_z = z_scores[:, 1]

# Count distances with absolute Z-score > 2
print("Distances with |Z| > 2:", np.sum(np.abs(distance_z) > 2))


Standard deviations: [ 16.65   6.16 183.71   8.73]
Distances with |Z| > 2: 53


In [25]:

# Identify appointments booked more than 14 days ahead
long_lead = df["lead_days"] > 14

# Count these appointments
long_lead_count = long_lead.sum()

# Calculate their percentage
long_lead_pct = long_lead.mean() * 100

print("Appointments booked >14 days ahead:", long_lead_count)
print("Percentage:", round(long_lead_pct, 1), "%")


Appointments booked >14 days ahead: 296
Percentage: 29.9 %


In [26]:

# Load clinic information
clinics_path = "/content/drive/MyDrive/CareLoop_NoShow_Guard/data/clinics.csv"

clinics_df = pd.read_csv(clinics_path)

print("Clinic dataset shape:", clinics_df.shape)

display(clinics_df.head())


Clinic dataset shape: (8, 6)


,clinic_id,clinic_name,city,tier,latitude,longitude
0,C01,Kothrud Family Clinic,Pune,Metro,18.5074,73.8077
1,C02,Hadapsar Care Centre,Pune,Metro,18.5089,73.9260
2,C03,Sitabuldi Health Point,Nagpur,Tier-2,21.1458,79.0882
3,C04,Malviya Nagar Clinic,Jaipur,Tier-2,26.8553,75.8150
4,C05,Vaishali Nagar Clinic,Jaipur,Tier-2,26.9124,75.7873


In [27]:

import requests

# Select the first clinic
clinic = clinics_df.iloc[0]

url = "https://api.open-meteo.com/v1/forecast"

params = {
    "latitude": clinic["latitude"],
    "longitude": clinic["longitude"],
    "hourly": "temperature_2m",
    "forecast_days": 1
}

response = requests.get(url, params=params, timeout=10)

print("HTTP status code:", response.status_code)

data = response.json()

print("First hourly temperature:", data["hourly"]["temperature_2m"][0])


HTTP status code: 200
First hourly temperature: 17.8


In [29]:

import time

def get_temperature(latitude, longitude):
    url = "https://api.open-meteo.com/v1/forecast"

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "hourly": "temperature_2m",
        "forecast_days": 1
    }

    # Try a maximum of 2 times
    for attempt in range(2):
        try:
            response = requests.get(
                url,
                params=params,
                timeout=10
            )

            response.raise_for_status()

            data = response.json()
            temperatures = data.get("hourly", {}).get("temperature_2m", [])

            if temperatures and temperatures[0] is not None:
                return temperatures[0]

            return None

        except (requests.RequestException, ValueError, KeyError) as error:
            print(f"Attempt {attempt + 1} failed: {error}")

            if attempt == 0:
                time.sleep(2)

    return None


In [30]:

temperatures = []

for _, clinic in clinics_df.iterrows():
    temp = get_temperature(
        clinic["latitude"],
        clinic["longitude"]
    )

    temperatures.append(temp)

    print(clinic["clinic_name"], ":", temp)

    time.sleep(1)

clinics_df["temperature_2m"] = temperatures

print("\nTotal clinics:", len(clinics_df))
print("Temperatures retrieved:", clinics_df["temperature_2m"].notna().sum())


Kothrud Family Clinic : 17.8
Hadapsar Care Centre : 18.2
Sitabuldi Health Point : 18.2
Malviya Nagar Clinic : 16.0
Vaishali Nagar Clinic : 16.0
Edappally Medical Centre : 25.2
Vijay Nagar Clinic : 17.0
Gomti Nagar Care Point : 15.3

Total clinics: 8
Temperatures retrieved: 8


In [31]:

# Save enriched clinic data to Google Drive
output_path = "/content/drive/MyDrive/CareLoop_NoShow_Guard/clinics_enriched.csv"

clinics_df.to_csv(output_path, index=False)

print("Enriched clinic data saved successfully!")
print("Dataset shape:", clinics_df.shape)


Enriched clinic data saved successfully!
Dataset shape: (8, 7)


In [32]:

# Check the number of rows and columns
print("Dataset shape:", df.shape)

# Check remaining missing values
print("\nTotal missing values:", df.isna().sum().sum())

# Check duplicate appointment IDs
print("Duplicate appointment IDs:", df["appointment_id"].duplicated().sum())

# Display column names
print("\nColumns:")
print(df.columns.tolist())


Dataset shape: (990, 16)

Total missing values: 0
Duplicate appointment IDs: 0

Columns:
['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status', 'lead_days', 'no_show']


In [33]:

# Define the required column order
column_order = [
    "appointment_id",
    "patient_id",
    "clinic_id",
    "booked_on",
    "appt_date",
    "age",
    "gender",
    "distance_km",
    "fee_inr",
    "booking_channel",
    "visit_type",
    "reminder_sent",
    "prior_no_shows",
    "status",
    "lead_days",
    "no_show"
]

# Arrange columns and sort by appointment ID
df = df[column_order].sort_values("appointment_id").reset_index(drop=True)

# Save cleaned dataset
output_path = "/content/drive/MyDrive/CareLoop_NoShow_Guard/appointments_clean.csv"

df.to_csv(
    output_path,
    index=False,
    date_format="%Y-%m-%d"
)

print("Cleaned appointments exported successfully!")
print("Rows:", len(df))
print("Columns:", len(df.columns))


Cleaned appointments exported successfully!
Rows: 990
Columns: 16


In [34]:

print("CLINIC COLUMNS:")
print(clinics_df.dtypes)

print("\nAPPOINTMENT COLUMNS:")
print(df.dtypes)

print("\nFIRST CLINIC RECORD:")
print(clinics_df.iloc[0].to_dict())



CLINIC COLUMNS:
clinic_id          object
clinic_name        object
city               object
tier               object
latitude          float64
longitude         float64
temperature_2m    float64
dtype: object

APPOINTMENT COLUMNS:
appointment_id             object
patient_id                 object
clinic_id                  object
booked_on          datetime64[ns]
appt_date          datetime64[ns]
age                         int64
gender                     object
distance_km               float64
fee_inr                     int64
booking_channel            object
visit_type                 object
reminder_sent              object
prior_no_shows              int64
status                     object
lead_days                   int64
no_show                     int64
dtype: object

FIRST CLINIC RECORD:
{'clinic_id': 'C01', 'clinic_name': 'Kothrud Family Clinic', 'city': 'Pune', 'tier': 'Metro', 'latitude': 18.5074, 'longitude': 73.8077, 'temperature_2m': 17.8}


In [35]:

def sql_value(value):
    if pd.isna(value):
        return "NULL"

    if isinstance(value, pd.Timestamp):
        return f"'{value.strftime('%Y-%m-%d')}'"

    if isinstance(value, str):
        return "'" + value.replace("'", "''") + "'"

    if isinstance(value, (int, float, np.integer, np.floating)):
        return str(value)

    raise TypeError(f"Unsupported value type: {type(value)}")


In [36]:

sql_path = "/content/drive/MyDrive/CareLoop_NoShow_Guard/load_data.sql"

# Use the original six clinic columns.
# Temperature enrichment is kept in clinics_enriched.csv.
clinic_columns = [
    "clinic_id", "clinic_name", "city",
    "tier", "latitude", "longitude"
]

appointment_columns = df.columns.tolist()

with open(sql_path, "w", encoding="utf-8") as file:

    file.write("-- CareLoop Clinics: Data Loading\n")
    file.write("-- Run after creating the database tables.\n\n")

    file.write("-- Insert clinic records first\n")

    for _, row in clinics_df.iterrows():
        values = ", ".join(sql_value(row[col]) for col in clinic_columns)
        columns = ", ".join(clinic_columns)

        file.write(
            f"INSERT INTO clinics ({columns}) VALUES ({values});\n"
        )

    file.write("\n-- Insert appointment records\n")

    for _, row in df.iterrows():
        values = ", ".join(sql_value(row[col]) for col in appointment_columns)
        columns = ", ".join(appointment_columns)

        file.write(
            f"INSERT INTO appointments ({columns}) VALUES ({values});\n"
        )

print("SQL file generated successfully!")
print("Expected INSERT statements:", len(clinics_df) + len(df))


SQL file generated successfully!
Expected INSERT statements: 998


In [37]:

with open(sql_path, "r", encoding="utf-8") as file:
    sql_content = file.read()

print("Total INSERT statements:", sql_content.count("INSERT INTO"))
print("\nFirst 500 characters:\n")
print(sql_content[:500])


Total INSERT statements: 998

First 500 characters:

-- CareLoop Clinics: Data Loading
-- Run after creating the database tables.

-- Insert clinic records first
INSERT INTO clinics (clinic_id, clinic_name, city, tier, latitude, longitude) VALUES ('C01', 'Kothrud Family Clinic', 'Pune', 'Metro', 18.5074, 73.8077);
INSERT INTO clinics (clinic_id, clinic_name, city, tier, latitude, longitude) VALUES ('C02', 'Hadapsar Care Centre', 'Pune', 'Metro', 18.5089, 73.926);
INSERT INTO clinics (clinic_id, clinic_name, city, tier, latitude, longitude) VALUES 
